# Kratt — LLM per-comment labeling

Labels each individual YouTube comment as **genuine / copycat / low-effort / ads_spam** using a
local LLM, and exports the original dataset with that tag added as a new column
(`labeled_comments.csv`). This notebook does **not** train a model -- it only produces labeled data.

**Team workflow:** the full dataset is pre-split into ~10,000-row part CSVs, one per team member.
Attach YOUR part file as this notebook's input dataset -- the notebook labels the **entire attached
file** (there is no sampling). Progress is checkpointed, so a session timeout just means re-running.

### ⚠ Why this exists — the label caveat
`niche_tag` (in the raw scraped data) is assigned **per video**, not per comment. All ~127k
comments come from **31 videos**, so `niche_tag` is only 31 independent label decisions --
far too coarse to train a per-comment classifier on directly. This notebook's LLM pass gives
every individual comment its own judgment instead, which is the real per-comment training
signal a future model would need.

## Setup
**Hosted notebook (Colab / Kaggle):** run this cell once, then if it installs anything for the
first time this session, **restart the kernel** (Colab: Runtime -> Restart session; Kaggle: Run ->
Restart Session) before running the rest of the notebook. Re-running this cell alone will NOT fix
a torch/transformers version-mismatch error -- the host keeps the old modules cached in memory
until the session actually restarts.

In [ ]:
import os, sys
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or os.path.exists('/kaggle')
HOSTED    = IN_COLAB or IN_KAGGLE

if HOSTED:
    # Colab and Kaggle both ship a torch build already matched to their GPU driver/CUDA
    # version. Do NOT pip install a different torch/transformers here -- doing that into a
    # live kernel leaves the ORIGINAL versions cached in sys.modules while the new ones sit
    # on disk, which is exactly what produces:
    #   AttributeError: module 'torch' has no attribute '_utils'
    #   ModuleNotFoundError: Could not import module 'Trainer'
    # Only add the small extras this notebook needs; leave torch/transformers alone.
    !pip install -q emoji accelerate bitsandbytes wordfreq
    print(f"{'Kaggle' if IN_KAGGLE else 'Colab'}: installed missing extras only "
          "(torch/transformers untouched).")
else:
    # Local venv: install exact pinned versions once, matching backend/requirements.txt.
    # !pip install -r ../requirements.txt
    pass

# Windows-only note: torch and MKL/numpy each bundle an OpenMP runtime (libiomp5md.dll);
# if a 2nd one loads it can abort the process with no traceback. No-op on Colab/Kaggle/Linux.
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Kaggle/Colab images often ship hf_transfer, a Rust-based accelerated HF Hub downloader that
# is auto-enabled when present. It has a known failure mode: large files silently stall
# partway through with no error. This MUST be set before transformers/huggingface_hub is
# imported anywhere (it's read once as a module constant, not re-checked per download).
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

import faulthandler; faulthandler.enable()  # native crash -> real C stack on stderr, not a silent death
import re, json, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('python:', sys.executable)
print('torch', torch.__version__, '| numpy', np.__version__, '| device', DEVICE,
      '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')

# quick import-health check for the exact failure this notebook hit before -- fails loudly
# and early (with a clear message) instead of cascading into a confusing error later
try:
    from transformers import AutoTokenizer  # noqa: F401
    print('transformers import OK')
except Exception as e:
    raise RuntimeError(
        'transformers/torch import is broken in this kernel session -- most likely a version '
        'mismatch left over from a previous pip install. Restart the kernel (Colab: Runtime -> '
        'Restart session; Kaggle: Run -> Restart Session), then Run All from the top -- do not '
        're-run cells in place.'
    ) from e

# --- config ---
if IN_KAGGLE:
    # Each team member attaches their own pre-split part CSV (any filename). Preference order:
    # a file dropped in /kaggle/working, the project's known full-dataset path, any raw_comments.csv,
    # then ANY csv under /kaggle/input -- with one dataset attached per run this resolves uniquely.
    _known = Path('/kaggle/input/datasets/geraldadli/krattt/raw_comments.csv')
    _candidates  = sorted(Path('/kaggle/working').glob('*.csv'))
    _candidates += [_known]
    _candidates += sorted(Path('/kaggle/input').rglob('raw_comments.csv'))
    _candidates += sorted(p for p in Path('/kaggle/input').rglob('*.csv')
                          if 'llm_labels' not in p.name and 'labeled_comments' not in p.name)
    DATA_PATH = next((p for p in _candidates if p.exists()), _known)
    if not DATA_PATH.exists():
        print('No input CSV found under /kaggle/input or /kaggle/working.')
        print("Attach your part file via the notebook's right sidebar -> Add Input -> Upload Dataset.")
    else:
        print(f'ATTACHED INPUT RESOLVED TO: {DATA_PATH} -- confirm this is YOUR part file.')
elif IN_COLAB:
    _candidates = [Path('/content/raw_comments.csv'),
                   Path('/content/drive/MyDrive/raw_comments.csv')]
    DATA_PATH = next((p for p in _candidates if p.exists()), _candidates[0])
    if not DATA_PATH.exists():
        print('raw_comments.csv not found on Colab. Either:')
        print('  (a) left sidebar -> Files -> upload it to /content, or')
        print('  (b) mount Drive and place it in MyDrive:')
        print("      from google.colab import drive; drive.mount('/content/drive')")
else:
    DATA_PATH = Path(r'C:\Users\Asus\Documents\Big Four\Kratt2\raw_comments.csv')
    if not DATA_PATH.exists():
        DATA_PATH = Path('../../../raw_comments.csv').resolve()  # backend/notebooks/ -> repo parent

# HF Hub downloads on hosted notebooks occasionally stall mid-transfer (connection drops but
# never errors, progress bar just freezes). Give stalled reads a timeout so they fail and
# retry instead of hanging forever.
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '60')

def load_with_retry(loader, *args, retries=4, **kwargs):
    for attempt in range(1, retries + 1):
        try:
            return loader(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                raise
            print(f'load attempt {attempt}/{retries} failed ({e!r}) -- retrying in {5*attempt}s')
            time.sleep(5 * attempt)

SEED       = 42
OUTPUT_DIR = Path('/kaggle/working/bert_kratt_out') if IN_KAGGLE else Path('bert_kratt_out')
np.random.seed(SEED); torch.manual_seed(SEED)
print('data:', DATA_PATH)

## O — Obtain
Already collected via the YouTube Data API scraper. Just load and confirm the shape + label provenance.

In [ ]:
df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
display(df.head(3))
print('\nclass counts:'); print(df['niche_tag'].value_counts())
print('\nunique videos:', df['video_id'].nunique())
print('videos per class:'); print(df.groupby('niche_tag')['video_id'].nunique())
# sanity: every video is single-label (distant supervision)
assert (df.groupby('video_id')['niche_tag'].nunique() == 1).all(), 'a video spans multiple labels?!'
print('\nOK: every video has exactly one niche_tag (video-level labels).')

## S — Scrub
- **Demojize** so emoji survive tokenization as text (😂 → `:face_with_tears_of_joy:`) instead of `[UNK]`.
- **Intra-video de-duplication** removes spam repetition without creating cross-video label conflicts; keep the count as a signal.
- Drop empties and truncate pathological giant comments (max was ~10k chars).

In [ ]:
import emoji

def clean_text(s):
    s = emoji.demojize(str(s))            # colon-delimited tokens, reversible
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df['clean_text'] = df['text'].map(clean_text)
df = df[df['clean_text'].str.len() > 0].copy()

# intra-video exact duplicates -> keep one, remember how many there were
df['dup_count'] = df.groupby(['video_id', 'clean_text'])['comment_id'].transform('size')
before = len(df)
df = df.drop_duplicates(subset=['video_id', 'clean_text'], keep='first').copy()
print(f'dropped {before - len(df)} intra-video duplicate rows -> {len(df)} remain')

df['char_len'] = df['clean_text'].str.len()
df = df[df['char_len'] <= 2000].copy()     # drop giant outliers
df = df.reset_index(drop=True)
print('final rows:', len(df))

## Lexical signal — top-100 words + dictionary check
Team heuristic: **a comment using NONE of its video's most common words is an outlier** -- either
an original/genuine thought, or plain gibberish. A dictionary check splits those two.
Each comment gets one `lex_hint` value the LLM receives as metadata:
- `common` -- shares at least one of the video's top-100 words (the normal case)
- `original` -- zero top-100 words, but real language -> lean genuine
- `gibberish?` -- zero top-100 words AND fails the dictionary check -> lean low-effort

The dictionary is **multilingual** (wordfreq, 12 Latin-script languages): ~40% of this data is
non-English, and Spanish/Indonesian etc. are written in plain ASCII -- an English-only wordlist
would brand those real comments as gibberish. Non-Latin scripts (CJK, Devanagari, Arabic) are
never marked gibberish -- they stay `original` and the multilingual LLM judges the text itself.

In [ ]:
from collections import Counter
from functools import lru_cache
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from wordfreq import zipf_frequency

# Latin-script languages to check words against. English alone is NOT enough: Spanish,
# Indonesian, Portuguese etc. are written in plain ASCII, and an English-only wordlist would
# brand real comments in those languages as gibberish (~40% of this data is non-English).
LEX_LANGS = ['en', 'es', 'pt', 'id', 'fr', 'de', 'it', 'tr', 'nl', 'pl', 'ms', 'vi']

@lru_cache(maxsize=200000)
def is_known_word(w):
    for lang in LEX_LANGS:
        try:
            if zipf_frequency(w, lang) > 0:
                return True
        except Exception:
            continue
    return False

def lex_tokens(text):
    # >=3 chars and non-stopword -- otherwise 'the/is/a' dominates every top-100 and the signal dies
    return [w for w in re.findall(r'\w+', str(text).lower())
            if len(w) >= 3 and w not in ENGLISH_STOP_WORDS]

def lex_hint_for(tokens, top100):
    if not tokens:
        return 'common'   # no usable tokens -> no evidence of originality; stay neutral
    if any(w in top100 for w in tokens):
        return 'common'
    # outlier: shares nothing with the section's vocabulary -> original thought or gibberish.
    # Only ASCII alphabetic tokens can be confidently called gibberish -- scripts we can't
    # verify (CJK, Devanagari, Arabic, ...) stay 'original' and the multilingual LLM judges
    # the actual text itself.
    alpha = [w for w in tokens if w.isalpha()]
    ascii_alpha = [w for w in alpha if w.isascii()]
    if alpha and ascii_alpha and len(ascii_alpha) / len(alpha) >= 0.7:
        known = sum(is_known_word(w) for w in ascii_alpha)
        if known / len(ascii_alpha) < 0.5:
            return 'gibberish?'
    return 'original'

df['lex_hint'] = ''
for vid, vid_df in df.groupby('video_id'):
    token_lists = vid_df['clean_text'].map(lex_tokens)
    counts = Counter()
    for toks in token_lists:
        counts.update(toks)
    # min count 3: a word used once or twice is not a 'most used word'. Without this floor,
    # small comment sections (< 100 unique tokens) would put EVERY word -- including singleton
    # words from the very comment being judged -- into the top-100, and the signal degenerates
    # to 'common' for everything.
    top100 = {w for w, c in counts.most_common(100) if c >= 3}
    df.loc[vid_df.index, 'lex_hint'] = [lex_hint_for(toks, top100) for toks in token_lists]

print(df['lex_hint'].value_counts())

## L — LLM-labeled per-comment tags
**Design notes:**
- Runs entirely on this notebook's own GPU via `transformers` (a quantized open-weight instruct
  model) -- no API key, no per-call cost. A separate Ollama server process was deliberately
  avoided: Kaggle sessions are ephemeral containers, and standing up a background daemon here
  would introduce model-download fragility (stalls, version drift) for no benefit over loading
  the model directly in-process.
- Batches ~20 comments **from the same video** per call, so the model sees siblings together
  (useful for spotting near-duplicate/copy-paste patterns, backed by the `dup_count` computed in
  Scrub) and gets the video's niche as context -- the same domain reasoning used to define these
  categories in the first place.
- Each comment carries metadata the LLM is told how to use: `likes`, `replies`, `repeats`
  (intra-video duplicate count), and the `lex_hint` from the previous cell.
- The prompt encodes the team's observed bot patterns explicitly (template comments, ad bots,
  copy-pasted top comments, gibberish) with priority ordering -- a small model applies
  'first rule that fits' far more reliably than weighing four categories at once. It is also
  told NOT to speculate about commenter profiles (context-aware bots are out of scope for
  text-only judgment).
- **Labels the entire attached file** -- the team pre-splits the dataset into ~10k-row part CSVs,
  one per member, so no sampling happens here.
- **Checkpointed and resumable.** Every completed batch is written to disk immediately, and
  re-running the loop skips whatever is already labeled -- safe to interrupt and continue later,
  or to keep running across multiple Kaggle sessions if it doesn't finish in one.

In [ ]:
# Local open-weight instruct LLM, 4-bit quantized to comfortably fit a single GPU.
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# On Kaggle, load the copy already attached as a dataset (no HF Hub download of ~15GB, avoiding
# any download stall entirely). Falls back to the HF Hub id elsewhere, which downloads normally
# (subject to the retry/timeout guards from Setup).
_LOCAL_LLM = Path('/kaggle/input/notebooks/fishcat/qwen-qwen2-5-7b-instruct')
LLM_MODEL_NAME = str(_LOCAL_LLM) if (IN_KAGGLE and _LOCAL_LLM.exists()) else 'Qwen/Qwen2.5-7B-Instruct'
if IN_KAGGLE and not _LOCAL_LLM.exists():
    print(f'expected local LLM at {_LOCAL_LLM} but it is missing -- falling back to HF Hub download.')

_quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                    bnb_4bit_quant_type='nf4')

llm_tokenizer = load_with_retry(AutoTokenizer.from_pretrained, LLM_MODEL_NAME)
llm_model = load_with_retry(AutoModelForCausalLM.from_pretrained, LLM_MODEL_NAME,
                             quantization_config=_quant_config, device_map='auto')
llm_model.eval()
print('LLM loaded:', LLM_MODEL_NAME)

In [ ]:
# --- config ---
LLM_BATCH_SIZE = 20    # comments per call, grouped within the same video
LLM_LABELS     = ['genuine', 'copycat', 'low-effort', 'ads_spam']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LLM_CHECKPOINT = OUTPUT_DIR / 'llm_labels.csv'

LLM_PROMPT_TEMPLATE = '''You are labeling YouTube comments to detect bot-like behavior. This video's niche is "{niche}".
Give each comment exactly ONE label. Read the rules top to bottom; use the FIRST label that fits.

1. "ads_spam" -- advertisement bot:
- contains a link, phone number, or channel/handle promotion
- promotes crypto, gambling, earnings ("I make $5000/week"), Telegram/WhatsApp contact
- "check my page/channel" self-promotion

2. "copycat" -- copied text:
- repeats=N > 1 means this exact text appears N times in this video's comments: label ALL of these copycat
- same phrasing as a popular comment but with far fewer likes (someone copying a top comment)

3. "low-effort" -- generic or nonsense filler:
- would fit under ANY video without changing a word: "nice video! looking forward to your next uploads",
  "great content", "love this", "first!", "who's watching in 2026"
- emoji-only, or one or two generic words
- incoherent gibberish/noise that is not about any video content (hint=gibberish? marks likely cases)

4. "genuine" -- a real human reaction:
- mentions something specific from THIS video: a moment, a person, a detail, a joke, a question, a criticism
- an opinion that only makes sense for this particular video
- replies=N > 0 is a hint that real people engaged with it

How to use the numbers and hints:
- likes: high likes on a SPECIFIC comment = community valued it (lean genuine). High likes on a
  generic or promotional comment = possibly boosted (do not let likes excuse spam).
- replies: real conversations get replies; bots rarely do.
- repeats: greater than 1 = copied text in this comment section (lean copycat).
- hint=common: the comment uses this video's most frequent words -- judge it by rules 1-4 normally.
- hint=original: the comment uses NONE of this video's common words but is real language --
  an independent thought, lean genuine (rules 1-3 still win if they match).
- hint=gibberish?: no common words AND fails a dictionary check -- likely nonsense, lean low-effort.

Do NOT guess about the commenter's profile or posting history -- judge only the text, numbers, and hints shown.

Comments (id | likes | replies | repeats | hint | text):
{comments_block}

Respond with ONLY a JSON object mapping each comment id to its label, nothing else, e.g.:
{{"abc123": "genuine", "def456": "low-effort"}}'''

def build_comments_block(batch_df):
    lines = []
    for row in batch_df.itertuples():
        text = row.clean_text[:300]
        lines.append(f'- id={row.comment_id!r} likes={row.like_count} replies={row.reply_count} '
                     f'repeats={row.dup_count} hint={row.lex_hint}: {text!r}')
    return chr(10).join(lines)

def parse_llm_json(raw_text):
    match = re.search(r'\{.*\}', raw_text, re.DOTALL)
    if not match:
        raise ValueError(f'no JSON object found in LLM output: {raw_text[:200]!r}')
    return json.loads(match.group(0))

def label_batch(niche, batch_df):
    # Single attempt, no retries -- faster runtime. A comment that doesn't come back with a valid
    # id/label just gets skipped rather than retried; the rest of the batch is still kept.
    prompt = LLM_PROMPT_TEMPLATE.format(niche=niche, comments_block=build_comments_block(batch_df))
    expected_ids = set(batch_df['comment_id'])
    messages = [{'role': 'user', 'content': prompt}]
    # return_dict=True forces a deterministic {input_ids, attention_mask} dict back -- newer
    # transformers versions can otherwise return a BatchEncoding instead of a bare tensor here,
    # which generate() can't consume directly (a confusing AttributeError deep inside generate()
    # rather than a clear type error at this line).
    encoded = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True)
    encoded = {k: v.to(next(llm_model.parameters()).device) for k, v in encoded.items()}
    with torch.no_grad():
        out = llm_model.generate(**encoded, max_new_tokens=800, do_sample=False,
                                  pad_token_id=llm_tokenizer.eos_token_id)
    raw = llm_tokenizer.decode(out[0][encoded['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        parsed = parse_llm_json(raw)
    except Exception as e:
        print(f"Skipped comment that don't make sense (batch of {len(expected_ids)} failed to parse: {e!r})")
        return {}
    valid = {cid: lab for cid, lab in parsed.items() if cid in expected_ids and lab in LLM_LABELS}
    missing = expected_ids - valid.keys()
    if missing:
        print(f"Skipped comment that don't make sense ({len(missing)}/{len(expected_ids)} in this batch)")
    return valid

In [ ]:
# --- run the labeling pass (safe to interrupt and re-run -- picks up where it left off) ---
# A checkpoint file can EXIST but be empty/headerless if a previous run crashed right after
# opening it (open(path, 'a') creates the file before anything is ever written to it) -- so
# check for actual content, not just existence, before trusting it as a resume point.
def _checkpoint_has_data(path):
    return path.exists() and path.stat().st_size > 0

already_done = set()
if _checkpoint_has_data(LLM_CHECKPOINT):
    already_done = set(pd.read_csv(LLM_CHECKPOINT)['comment_id'])
    print(f'resuming: {len(already_done)} comments already labeled in a previous run')

work_df = df[~df['comment_id'].isin(already_done)]
print(f'{len(work_df)} comments left to label (entire attached file)')

start_t = time.time()
done_count = 0
total = len(work_df)
write_header = not _checkpoint_has_data(LLM_CHECKPOINT)

with open(LLM_CHECKPOINT, 'a', newline='', encoding='utf-8') as f:
    for video_id, vid_df in work_df.groupby('video_id'):
        niche = vid_df['niche_tag'].iloc[0]
        for i in range(0, len(vid_df), LLM_BATCH_SIZE):
            batch = vid_df.iloc[i:i + LLM_BATCH_SIZE]
            labels = label_batch(niche, batch)
            text_by_id = batch.set_index('comment_id')['text']
            rows = pd.DataFrame([{'comment_id': cid, 'text': text_by_id[cid], 'llm_label': lab}
                                 for cid, lab in labels.items()],
                                columns=['comment_id', 'text', 'llm_label'])
            if len(rows):
                rows.to_csv(f, header=write_header, index=False)
                write_header = False
                f.flush()
            done_count += len(batch)
            if done_count % (LLM_BATCH_SIZE * 10) < LLM_BATCH_SIZE or done_count >= total:
                elapsed = time.time() - start_t
                rate = done_count / elapsed if elapsed > 0 else 0
                eta_min = (total - done_count) / rate / 60 if rate > 0 else float('nan')
                print(f'{done_count}/{total} comments | {rate:.2f}/sec | ETA {eta_min:.0f} min')

print('done. checkpoint at', LLM_CHECKPOINT.resolve())

In [ ]:
# --- merge the checkpoint back in (works even if only a partial/limited run has happened so far) ---
# llm_labels.csv also carries a 'text' column (for easy manual QA of the checkpoint file itself);
# select only what's needed here so it doesn't collide with df's own 'text' column on merge.
if _checkpoint_has_data(LLM_CHECKPOINT):
    llm_labels_df = pd.read_csv(LLM_CHECKPOINT)[['comment_id', 'llm_label']]
else:
    llm_labels_df = pd.DataFrame(columns=['comment_id', 'llm_label'])

df = df.drop(columns=['llm_label'], errors='ignore').merge(llm_labels_df, on='comment_id', how='left')
df['label_source'] = np.where(df['llm_label'].notna(), 'llm', 'niche_tag')
df['effective_label'] = df['llm_label'].where(df['llm_label'].notna(), df['niche_tag'])

coverage = df['llm_label'].notna().mean()
print(f'LLM label coverage: {coverage*100:.1f}% of {len(df)} comments')
print(chr(10) + 'LLM label distribution:'); print(df['llm_label'].value_counts())
print(chr(10) + 'LLM label vs niche_tag (row-normalized -- sanity check: a video labeled genuine')
print('should skew toward llm_label==genuine, but with real per-comment variation, not 100/0):')
print(pd.crosstab(df['niche_tag'], df['llm_label'], normalize='index').round(2))

In [ ]:
# --- export: same original dataset, with an extra per-comment tag column ---
# Broadcasts each labeled (video_id, clean_text) group's tag back onto every row that shares it
# (e.g. duplicate spam/low-effort comments inherit their group's label), so every original
# comment_id gets a tag -- not just the deduped representative that was actually sent to the LLM.
df_original = pd.read_csv(DATA_PATH)
df_original['clean_text'] = df_original['text'].map(clean_text)

_label_by_group = df[['video_id', 'clean_text', 'effective_label', 'label_source']].drop_duplicates(
    subset=['video_id', 'clean_text'])
labeled_full = df_original.merge(_label_by_group, on=['video_id', 'clean_text'], how='left')
labeled_full = labeled_full.drop(columns=['clean_text']).rename(columns={'effective_label': 'comment_tag'})

EXPORT_PATH = OUTPUT_DIR / 'labeled_comments.csv'
labeled_full.to_csv(EXPORT_PATH, index=False)

n_tagged = labeled_full['comment_tag'].notna().sum()
print(f'exported {len(labeled_full)} rows (matches original {len(df_original)}-row dataset) -> '
      f'{EXPORT_PATH.resolve()}')
print(f'tagged: {n_tagged} / {len(labeled_full)} ({n_tagged/len(labeled_full)*100:.1f}%)')
print(chr(10) + 'comment_tag distribution:'); print(labeled_full['comment_tag'].value_counts())

## Next steps
1. **QA the tags** -- spot-check `labeled_comments.csv`, especially the niche_tag-vs-llm_label
   crosstab printed above: a video labeled `genuine` should skew toward `genuine` tags, but with
   real per-comment variation, not 100/0.
2. **Gold set:** hand-label ~200–300 comments yourself and compare against the LLM's tags on the
   same comments -- the only true check of whether the LLM is judging authenticity, not just
   parroting a language/topic shortcut.
3. **Train on it:** feed `comment_tag` into a separate BERT/XLM-R fine-tuning notebook as the
   per-comment label (replacing the coarse `niche_tag`). Keep `ads_spam` as the rule engine's job,
   not the classifier's, per the original hybrid design.
4. **Combine the parts:** once every team member's run finishes, concatenate the per-part
   `labeled_comments.csv` files into the full labeled dataset (drop nothing -- `comment_id` keys
   are globally unique, so a plain concat is safe).